# 09 — Familia F5: Volatilidad condicional (GARCH y MS-GARCH)

**Qué detecta.** Regímenes de **volatilidad** del S&P 500 modelando explícitamente la
varianza condicional $\sigma_t^2$ (heterocedasticidad). La idea rectora es que el estrés
de mercado se manifiesta antes y con más nitidez en la varianza que en la media, y que la
varianza condicional de un GARCH **reacciona el mismo día** a un shock grande.

**Supuestos.** (i) La media condicional es casi impredecible y la estructura explotable
está en la varianza; (ii) $\sigma_t^2$ es función determinista del pasado observable
(causal por construcción); (iii) innovaciones estandarizadas i.i.d. con colas gruesas
(t de Student); (iv) **univariante**: solo el retorno del S&P 500.

**Detectores de la familia.**

| id | detector | módulo | cómo obtiene el régimen |
|---|---|---|---|
| D06 | GJR-GARCH(1,1)-t + umbral de σ | `regimenes.detectores.f5_garch.garch_t_vol` | umbral con histéresis y *dwell* sobre la σ condicional (vía A) |
| D11 | MS-GARCH(1,1)-t de Haas-Mittnik-Paolella | `regimenes.detectores.f5_garch.msgarch_regime` | probabilidad filtrada de Hamilton de un régimen latente con GARCH propio (vía B) |

D06 es el **baseline de volatilidad barato y robusto**; D11 es la **síntesis** de F4
(régimen de Markov latente) y F5 (dinámica GARCH dentro de cada régimen), exploratoria
por coste y fragilidad de estimación.

**Regla del notebook.** Ninguna cifra del texto está escrita a mano: todas salen de las
celdas. Por defecto el notebook no ajusta el benchmark: lee la caché verificada de
`results/benchmark/` (§3). Los ajustes de muestra completa de §4 son **ilustrativos y
no causales**; la evaluación honesta es la OOS walk-forward del benchmark.

**Índice**

1. [Teoría de la familia](#s1)
2. [Configuración (desde el registro)](#s2)
3. [Ejecución y carga de resultados](#s3)
4. Detectores por pista
   - [4.1 D06 GJR-GARCH-t](#s4-1): estados OOS · σ condicional y umbrales · curva de
     impacto de noticias · colas-t (QQ) · σ frente a volatilidad realizada · cobertura ·
     métricas y ranking
   - [4.2 D11 MS-GARCH](#s4-2): estados OOS y probabilidad filtrada · degeneración
     fold a fold · causa raíz · ajuste de muestra completa (opcional) · cobertura ·
     métricas y ranking
5. [Comparación intra-familia (D06 vs D11)](#s5)
6. [Hallazgos de la Capa 1 (v1) y qué se mantiene / cambia en v2](#s6)
7. [Fortalezas, limitaciones y conclusión](#s7)

<a id="s1"></a>
## 1. Teoría de la familia

*Fuentes: `docs/teoria/F5_volatilidad_garch.md` (antes `memory/sota/05_volatilidad_garch.md`),
`docs/teoria/00_estado_del_arte.md`, `docs/detectores/D06_garch_t_vol.md`,
`docs/detectores/D11_msgarch_regime.md` y los notebooks v1
`capa1-final:capa1_exploracion/notebooks/06_garch_t_vol.ipynb` y `11_msgarch_regime.ipynb` (tag git
`capa1-final`; cifras v1 en `docs/historia/capa1/resultados/`). Claves de
`docs/references.bib` entre corchetes.*

### 1.1 De ARCH a GARCH

Los modelos de la familia ARCH descomponen el retorno en media y una innovación con
varianza condicional variable:

$$
r_t = \mu + \varepsilon_t,\qquad \varepsilon_t = \sigma_t z_t,\qquad z_t \overset{iid}{\sim} D(0,1).
$$

ARCH [vol_engle1982] hace $\sigma_t^2$ función de los cuadrados de las innovaciones
pasadas y captura el **clustering de volatilidad**; GARCH(1,1) [vol_bollerslev1986] añade
la propia varianza pasada, una representación parsimoniosa de memoria larga:

$$
\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2,\qquad
\omega>0,\ \alpha,\beta\ge 0,\ \alpha+\beta<1.
$$

La varianza incondicional es $\bar\sigma^2=\omega/(1-\alpha-\beta)$ y la **persistencia**
$\alpha+\beta$ fija la vida media de un shock, $h_{1/2}=\ln 0{,}5/\ln(\alpha+\beta)$.
Como $\sigma_t^2$ solo usa información hasta $t-1$, el modelo es **causal por
construcción**: ideal para un marco walk-forward.

### 1.2 Asimetría (GJR) y colas (t de Student)

**Efecto apalancamiento.** Las caídas elevan la volatilidad futura más que las subidas
del mismo tamaño. GJR-GARCH [vol_glosten1993] lo captura con un indicador de signo:

$$
\sigma_t^2 = \omega + \big(\alpha + \gamma\,\mathbb 1[\varepsilon_{t-1}<0]\big)\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2,
$$

con persistencia $\alpha+\beta+\gamma/2$ bajo innovaciones simétricas. Alternativas:
EGARCH [vol_nelson1991] (log-varianza, sin restricciones de signo) y TGARCH
[vol_zakoian1994] (asimetría sobre la desviación típica). La **curva de impacto de
noticias** (§4.1) dibuja $\sigma_t$ frente a $\varepsilon_{t-1}$ con
$\sigma_{t-1}^2=\bar\sigma^2$: la rama negativa por encima de la positiva es $\gamma>0$.

**Colas gruesas.** Un GARCH gaussiano absorbe el clustering pero deja residuos
estandarizados leptocúrticos. GARCH-t [vol_bollerslev1987] usa
$z_t\sim t_\nu\sqrt{(\nu-2)/\nu}$ (varianza unidad), con curtosis en exceso $6/(\nu-4)$
para $\nu>4$. El QQ-plot de los residuos (§4.1) es la prueba visual.

### 1.3 Vía A: del nivel de σ a un régimen (D06)

GARCH **no detecta régimen**: produce una serie continua $\sigma_t$. D06 añade una capa de
decisión, un autómata de dos estados con **histéresis** y **permanencia mínima**:

$$
\text{entra en crisis si } \sigma_t>\tau_{in};\qquad
\text{sale si } \sigma_t<\tau_{out}\ \text{y han pasado al menos } d_{min} \text{ sesiones},
$$

con $\tau_{in}>\tau_{out}$ percentiles de la σ del *train* (los cuantiles y $d_{min}$ son
los defaults del constructor, tabla de §2). La banda muerta $[\tau_{out},\tau_{in}]$ evita
el parpadeo cuando σ oscila alrededor del corte. `p_crisis` es una **sigmoide monótona**
de $(\sigma_t-\tau_{in})$, no una probabilidad posterior.

**Causalidad en el walk-forward.** `fit` estima por ML (librería `arch`
[vol_sheppard_arch], retornos ×100) y **congela** los parámetros y los umbrales. Para cada
bloque de test no se reestima: se reconstruye el modelo sobre
`[burn-in del train] + bloque`, se fijan los parámetros (`.fix`) y se lee la σ
condicional del bloque. En v1 se verificó que ocultar el futuro no cambia la σ del bloque.

### 1.4 Vía B: cambio de régimen dentro del GARCH (RS/MS-GARCH, D11)

SWARCH [vol_hamiltonsusmel1994] escala un ARCH por un factor de régimen markoviano. Dejar
un GARCH completo dentro de cada régimen choca con la **dependencia de la trayectoria**:
$\sigma_{t-1}^2$ depende del régimen de $t-1$, que depende del de $t-2$…, y la
verosimilitud integra sobre $K^t$ trayectorias. Soluciones:

- **Gray** [vol_gray1996]: colapsa $\sigma_{t-1}^2$ a su esperanza sobre las
  probabilidades de régimen.
- **Klaassen** [vol_klaassen2002]: colapsa condicionando además al régimen actual.
- **Haas-Mittnik-Paolella (HMP)** [vol_haasmittnikpaolella2004]: $K$ recursiones GARCH
  **paralelas**, cada una alimentada solo por su propia varianza pasada,

$$
h_{k,t} = \omega_k + \alpha_k\,\varepsilon_{t-1}^2 + \beta_k\,h_{k,t-1},\qquad
y_t\mid (S_t=k) \sim \mu + \sqrt{h_{k,t}}\;t_\nu(\mathrm{Var}=1),
$$

  sin dependencia de la trayectoria: la verosimilitud se calcula con el **filtro de
  Hamilton** (§1.2 del notebook 08) en $O(TK^2)$ y el posterior
  $P(S_t=\text{crisis}\mid y_{\le t})$ es causal.

D11 implementa HMP **desde cero** (numpy/scipy, sin R): $\mu$ única y $\nu$ compartido por
parsimonia; lo que separa los regímenes es la dinámica $(\omega_k,\alpha_k,\beta_k)$. El
estándar de referencia es el paquete `MSGARCH` de R [vol_ardia2019]; Marcucci
[vol_marcucci2005] mostró que el MRS-GARCH domina al GARCH en la previsión de volatilidad
del S&P 500 a horizontes cortos.

**Alcance (honesto).** D11 es la especificación **paralela / path-independiente** de HMP. No
reproduce el MS-GARCH path-dependiente completo (intratable), ni los colapsos de Gray o
Klaassen, ni las variantes del paquete `MSGARCH` de R (eGARCH o GJR por régimen, estimación
bayesiana): una sola familia, GARCH(1,1) por régimen con innovación t y $
u$ compartido.

### 1.5 Precio de la expresividad: identificación

El ML del MS-GARCH es **no convexo** (óptimos locales, *label switching*) y, con pocos
arranques y ventanas cortas, puede converger a soluciones con **dos regímenes casi
idénticos** o con uno de ellos nunca visitado. Entonces el filtro no conmuta y el
etiquetado económico por fold (que debe asignar "calma"/"crisis" a algo) se vuelve
arbitrario. Es el resultado negativo de v1 (§6) y lo que el diagnóstico fold a fold de
§4.2 examina en v2.

### 1.6 Relación con otras familias

- **F4 Markov-Switching (D05)**: régimen latente con varianza **constante** dentro del
  régimen; D11 = D05 + GARCH intra-régimen; D06 = GARCH sin régimen latente.
- **F1 reglas (D01)**: D06 es el mismo autómata de histéresis que la regla de
  volatilidad/VIX, pero sobre la σ condicional **endógena** del S&P 500 en lugar de un
  nivel observado.
- **Multivariante**: DCC-GARCH quedó fuera del núcleo (no etiqueta régimen de forma
  nativa); la correlación acciones-bonos se trata como feature en otras familias.

### Referencias

`vol_engle1982` · `vol_bollerslev1986` · `vol_bollerslev1987` · `vol_nelson1991` ·
`vol_glosten1993` · `vol_zakoian1994` · `vol_hamiltonsusmel1994` · `vol_gray1996` ·
`vol_klaassen2002` · `vol_haasmittnikpaolella2004` · `vol_marcucci2005` · `vol_ardia2019` ·
`vol_sheppard_arch` · `hamilton1989` (todas en `docs/references.bib`).

<a id="s2"></a>
## 2. Configuración (desde el registro)

La tabla se genera desde el registro, que es la única fuente de verdad de la
configuración del benchmark: features de entrada, parámetros pasados al constructor,
semilla, frecuencia de reajuste (`step`), tamaño del entrenamiento inicial y tipo de
ventana por pista. La fila `params_por_defecto` muestra los parámetros que el
registro **no** fija y que, por tanto, toman el valor por defecto de la clase (p. ej.
los umbrales de histéresis de D06). Las configuraciones son hipótesis previas al
benchmark: no se ajustaron mirando sus métricas.

In [ ]:
%matplotlib inline
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from regimenes import informes as inf
from regimenes import rutas, viz
from regimenes.benchmark import DEFAULT_TRAIN_DAYS, configure_evaluation
from regimenes.detectores.registry import SEED
from regimenes.evaluacion import ranking as rk
from regimenes.evaluacion.walk_forward import CONTEXT_YEARS
from regimenes.features import TRADING_DAYS

# --- Configuración común a los notebooks de familia 05–11 ---------------------------------
FAMILIA = 'f5_garch'                  # figuras en results/detectores/<FAMILIA>/
NOMBRE_FAMILIA = 'F5 — Volatilidad condicional (GARCH)'
DETECTORES = ['D06', 'D11']
PISTAS = ['A', 'B']
OUTPUT_DIR = rutas.RESULTS_BENCHMARK        # caché del benchmark (aquí solo se lee)
FIG_DIR = inf.carpeta_figuras(FAMILIA)
SPECS = inf.specs_familia(DETECTORES, PISTAS)   # {(pista, id): DetectorSpec} del registro
MIN_RUN = int(rk.DETECTION_DEFAULTS['min_run'])
BETA = float(rk.DETECTION_DEFAULTS['beta'])
rel = inf.ruta_relativa

viz.use_house_style()
inf.opciones_pandas()
warnings.filterwarnings('ignore', category=FutureWarning)


def guardar(fig, nombre):
    """Guarda en results/detectores/<FAMILIA>/<nombre>.png con el estilo de casa (convención 05–11)."""
    return inf.guardar_figura(fig, FIG_DIR, nombre)


print(f'Familia {NOMBRE_FAMILIA}: {", ".join(DETECTORES)} | pistas {", ".join(PISTAS)}')
print('Resultados benchmark :', rel(OUTPUT_DIR))
print('Figuras de la familia:', rel(FIG_DIR))

In [ ]:
CONFIG = inf.tabla_configuracion(SPECS)
CONFIG['n_parametros_modelo'] = [SPECS[(t, d)].factory().n_parameters() for d, t in CONFIG.index]
display(CONFIG.T)
print(f'Semilla global del registro: SEED = {SEED}; train inicial por pista (sesiones): {DEFAULT_TRAIN_DAYS}; '
      f'contexto de propagación de estado entre bloques (ADR-003): {CONTEXT_YEARS:g} año(s).')

<a id="s3"></a>
## 3. Ejecución y carga de resultados

`EJECUTAR = False` (por defecto) **no ajusta nada**: `inf.cargar_resultados_familia` lee la caché
del benchmark con `run_benchmark(cache_only=True)`, que solo acepta una combinación pista/detector si
su huella (código del paquete + datos procesados + especificación + entorno) coincide con la
registrada y el panel OOS existe. Si alguna combinación no está vigente, la celda se detiene con el
comando exacto a lanzar desde la raíz del repositorio (`python -m regimenes.benchmark --track A B
--detector …`). `EJECUTAR = True` pasa el gate previo y reajusta los detectores de esta familia en las
dos pistas con `run_job` (mismo camino que la CLI por detector: escribe `metrics/`, `panels/` y
`status/` de esas combinaciones y no toca `manifest.json` ni `run_status.csv`).

El puesto en el **ranking de detección ADR-003** se lee del `ranking*.csv` que genera
`12_comparativa.ipynb`, comprobando que sus columnas `det_*` coinciden con las métricas verificadas
(si el benchmark se re-ejecutó y 12 no, la celda pide regenerarlo). La segunda celda carga los datos
de apoyo (paneles procesados de cada pista, rejilla común, precio del S&P 500 y ventanas del juez),
que no dependen de `results/`.

In [ ]:
EJECUTAR = False   # True = reajusta los detectores de la familia con el protocolo del benchmark (ver 'lento' en §2)
FORZAR = False     # True = ignora la caché vigente al reajustar (solo con EJECUTAR = True)
COMANDO = inf.comando_benchmark(PISTAS, DETECTORES)

if EJECUTAR:
    estados_run = inf.ejecutar_familia(DETECTORES, PISTAS, output_dir=OUTPUT_DIR, forzar=FORZAR)
    display(estados_run[['pista', 'id', 'estado', 'segundos', 'detalle']])
    print('Consolida después manifest.json/run_status.csv con:\n'
          '    python -m regimenes.benchmark --consolidate --track A B')

# Solo lectura de la caché verificada por huella + paneles OOS + ranking ADR-003.
# Si alguna combinación no está vigente, falla con el comando exacto (COMANDO).
RES = inf.cargar_resultados_familia(DETECTORES, PISTAS, output_dir=OUTPUT_DIR)
display(RES.estado_cache[['pista', 'id', 'estado', 'detalle']])
METRICAS = RES.metricas        # métricas verificadas, índice (pista, id)
PANELES = RES.paneles          # {(pista, id): panel OOS con state, p_crisis, fold}
RANKING = RES.ranking          # ranking de detección ADR-003 completo (todas las familias)
RANK = RES.rank                # el mismo ranking indexado por (pista, id)
N_PISTA = RES.n_por_pista      # detectores por pista en el ranking
print(f'Métricas verificadas: {len(METRICAS)} combinaciones pista/detector.')
print(f'Ranking: {rel(RES.ruta_ranking)} (coherente con la caché) · detectores por pista: {N_PISTA}')
print('Parámetros del criterio ADR-003:', rk.DETECTION_DEFAULTS)

### 3.1 Datos de apoyo y utilidades de figura

Para las figuras hacen falta el precio del S&P 500 (espina cruda de `data/raw`) y, para
los diagnósticos ilustrativos, la misma matriz `X` que ve el benchmark (panel de la
pista recortado al índice común de los doce detectores, `X_PISTA`). Las utilidades de
figura y tabla son las comunes a los notebooks 05–11 (`regimenes.informes`): estados OOS
sobre el S&P 500, cobertura por crisis y resumen del ranking ADR-003.

In [ ]:
CTX = inf.cargar_contexto_pistas(PISTAS)   # data/processed + S&P 500 crudo (no depende de results/)
DATOS, INDICE, SP500 = CTX.datos, CTX.indice, CTX.sp500
CRISIS, TRAMPAS = CTX.crisis, CTX.trampas  # ventanas [pico, suelo] del juez por pista y trampas

for t in PISTAS:
    indices = [PANELES[(t, d)].index for d in DETECTORES]
    if not all(ix.equals(indices[0]) for ix in indices):
        raise RuntimeError(f'Pista {t}: los paneles OOS de la familia no comparten índice (gate de 12).')
    oos = indices[0]
    evaluables = [c for c, (a, b) in CRISIS[t].items() if pd.Timestamp(b) >= oos[0]]
    print(f'Pista {t}: OOS {oos[0].date()} -> {oos[-1].date()} ({len(oos)} sesiones); '
          f'{len(evaluables)} de {len(CRISIS[t])} crisis del catálogo con suelo dentro del OOS.')
print('Ventanas trampa (no-crisis):', ', '.join(f'{k} {a}->{b}' for k, (a, b) in TRAMPAS.items()))
X_PISTA = {(t, d): CTX.matriz(t, s.features) for (t, d), s in SPECS.items()}   # X del benchmark

# Utilidades comunes 05–11 (implementación en regimenes.informes; aquí solo se fijan los datos).
def estado_crisis(t, d):
    """Estado canónico de crisis del detector (n_states - 1)."""
    return RES.estado_crisis(t, d)


def es_crisis(t, d):
    """Serie booleana OOS: sesión en el estado de crisis."""
    return RES.es_crisis(t, d)


def tabla_cobertura(t, d):
    """Cobertura OOS por crisis (IC bootstrap por bloques), detección por evento y lead/lag."""
    return inf.tabla_cobertura(METRICAS.loc[(t, d)], CRISIS[t])


def tabla_trampas(t, d):
    """Activación en las ventanas trampa (fracción de sesiones marcadas; menor es mejor)."""
    return inf.tabla_trampas(METRICAS.loc[(t, d)], TRAMPAS)


def figura_estados(t, d, diagnostico=None, titulo_diag=None, **kw):
    """Arriba estados OOS sobre el S&P 500 con franjas de crisis/trampas; abajo el diagnóstico
    propio (``diagnostico(ax)``) o, por defecto, P(crisis) OOS del panel."""
    fig, axes = inf.figura_estados_oos(
        SP500, PANELES[(t, d)], RES.n_estados(t, d), crisis=CRISIS[t], trampas=TRAMPAS,
        titulo=f'{d} {SPECS[(t, d)].label} — pista {t}: estados OOS (causal) sobre el S&P 500',
        diagnostico=diagnostico, titulo_diagnostico=titulo_diag, **kw)
    guardar(fig, f'{d}_{t}_estados_oos')
    plt.show()
    return axes


def figura_cobertura(t, d, **kw):
    """Cobertura por crisis (rojo = detectada) y activación en trampas (gris) de una pista."""
    tab = tabla_cobertura(t, d)
    fig, _ = inf.figura_cobertura(
        tab, tabla_trampas(t, d), min_run=MIN_RUN,
        titulo=f'{d} — pista {t}: cobertura OOS por crisis (rojo = detectada, ≥{MIN_RUN} sesiones '
               'seguidas; barras = IC bootstrap; gris = trampas)', **kw)
    guardar(fig, f'{d}_{t}_cobertura_crisis')
    plt.show()
    return tab


def resumen_ranking(ids):
    """Métricas y puesto en el ranking de detección ADR-003 (columna 'de' = detectores en la pista)."""
    return RES.resumen_ranking(ids)


def franjas_crisis(ax, t, alpha=0.10):
    """Sombrea las ventanas [pico, suelo] de la pista (labels congeladas del spec)."""
    inf.franjas_eventos(ax, CRISIS[t], None, alpha_crisis=alpha)


COLS_METRICAS = ['n_oos', 'oos_start', 'oos_end', 'det_event_recall', 'det_n_detectados', 'det_n_eventos',
                 'det_precision', 'det_base_rate', 'det_lift_precision', 'det_marked_rate',
                 'det_mean_crisis_run', 'mean_crisis_coverage', 'false_alarm_rate', 'switching_rate',
                 'mean_regime_duration', 'label_stability', 'log_likelihood', 'aic', 'bic',
                 'elapsed_seconds']
COLS_RANKING = ['puesto_deteccion', 'nivel_etiqueta', 'score_deteccion', 'puesto_legacy']


def resumen_metricas(d):
    """Métricas verificadas + puesto ADR-003 de un detector (una columna por pista)."""
    filas = {}
    for t in PISTAS:
        serie = {c: METRICAS.loc[(t, d)].get(c, np.nan) for c in COLS_METRICAS}
        if 'mean_crisis_coverage' in RANK.columns:
            serie['mean_crisis_coverage'] = RANK.loc[(t, d), 'mean_crisis_coverage']
        serie.update({c: RANK.loc[(t, d)].get(c, np.nan) for c in COLS_RANKING})
        serie['n_detectores_en_pista'] = N_PISTA[t]
        filas[f'pista {t}'] = serie
    return pd.DataFrame(filas)

<a id="s4"></a>
## 4. Detector a detector

<a id="s4-1"></a>
### 4.1 D06 — GJR-GARCH(1,1)-t con umbral de σ

#### 4.1.1 Estados OOS sobre el S&P 500

Días marcados como crisis por el walk-forward causal (rojo), ventanas de crisis de la
pista (azul) y trampas (naranja). Abajo, la `p_crisis` OOS de D06: una sigmoide de
$(\sigma_t-\tau_{in})$ que vale 0,5 en el umbral de entrada (no es una probabilidad
posterior).

In [ ]:
for t in PISTAS:
    display(Markdown(f'#### Pista {t}'))
    figura_estados(t, 'D06', titulo_diag=f'D06 — pista {t}: sigmoide de (σ − τ_in) (causal, OOS)')
    print(f"D06/{t}: {float(es_crisis(t, 'D06').mean()):.1%} de las sesiones OOS marcadas como crisis; "
          f"{int((PANELES[(t, 'D06')]['state'].diff().fillna(0) != 0).sum())} cambios de estado.")

#### 4.1.2 Diagnóstico propio: σ condicional, umbrales y mecanismo del modelo

**Ajuste ilustrativo sobre la muestra completa (NO causal).** Una estimación por pista
sobre la misma `X` del benchmark (barata: un GARCH univariante) para leer los parámetros
($\omega,\alpha,\gamma,\beta,\nu$), la persistencia, la vida media de un shock, la
volatilidad incondicional y la curtosis implícita de la t, y para dibujar la σ con los
umbrales $\tau_{in}$/$\tau_{out}$. En el benchmark estos parámetros y umbrales se
reestiman en cada fold solo con el *train*.

In [ ]:
DIAGNOSTICO_ILUSTRATIVO = True  # un ajuste GJR-GARCH-t por pista sobre la muestra completa (segundos)

AJUSTES_D06 = {}
if DIAGNOSTICO_ILUSTRATIVO:
    filas = []
    for t in PISTAS:
        X = X_PISTA[(t, 'D06')]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            det = SPECS[(t, 'D06')].factory().fit(X)
            det.label_states_economically(X, market_returns=X['SP500_ret'])
        sigma = det._conditional_sigma(X)
        estados = pd.Series(det.predict(X), index=X.index).reindex(sigma.index)
        AJUSTES_D06[t] = {'det': det, 'sigma': sigma, 'estados': estados, 'X': X}
        p = det._params
        om, al, ga, be, nu = (float(p[k]) for k in ('omega', 'alpha[1]', 'gamma[1]', 'beta[1]', 'nu'))
        pers = al + be + 0.5 * ga
        sbar = om / (1.0 - pers) if pers < 1 else np.nan
        filas.append({
            'pista': t, 'muestra': f'{X.index.min().date()} → {X.index.max().date()}', 'sesiones': len(X),
            'mu': float(p['mu']), 'omega': om, 'alpha': al, 'gamma (leverage)': ga, 'beta': be, 'nu': nu,
            'persistencia α+β+γ/2': pers,
            'vida media shock (sesiones)': np.log(0.5) / np.log(pers) if 0 < pers < 1 else np.nan,
            'vol incondicional anualizada (%)': np.sqrt(sbar * TRADING_DAYS),
            'curtosis exceso t: 6/(ν−4)': 6.0 / (nu - 4.0) if nu > 4 else np.inf,
            'τ_in (σ ×100)': det._tau_in, 'τ_out (σ ×100)': det._tau_out,
            'q_in / q_out / dwell': f'{det.q_in} / {det.q_out} / {det.min_dwell}',
            'crisis = σ alta': bool(sigma[estados == det.crisis_state].mean() > sigma[estados != det.crisis_state].mean()),
            'logL': det.score(X), 'AIC': det.aic(X), 'BIC': det.bic(X),
        })
    PARAMS_D06 = pd.DataFrame(filas).set_index('pista')
    display(PARAMS_D06.T)

**(a) σ condicional con umbrales.** La σ (escala ×100 = volatilidad diaria en %) con
$\tau_{in}$ (entrada) y $\tau_{out}$ (salida) del ajuste de muestra completa, y las
ventanas de crisis de la pista.

In [ ]:
for t, a in AJUSTES_D06.items():
    det, sigma = a['det'], a['sigma']
    configure_evaluation(t)
    fig, ax = plt.subplots(figsize=(15, 4.5))
    ax.plot(sigma.index, sigma.values, color=viz.C_NEG, lw=0.6, label='σ condicional (×100)')
    ax.axhline(det._tau_in, color=viz.C_CRISIS, ls='--', lw=1.0, label=f'τ_in (percentil {det.q_in:g})')
    ax.axhline(det._tau_out, color=viz.C_SHORT, ls=':', lw=1.0, label=f'τ_out (percentil {det.q_out:g})')
    franjas_crisis(ax, t)
    ax.set_yscale('log'); ax.set_ylabel('σ diaria (%)')
    ax.set_title(f'D06 — pista {t}: σ condicional GJR-GARCH-t y umbrales (muestra completa, ilustrativo)')
    ax.legend(loc='upper left', framealpha=0.9)
    fig.tight_layout()
    guardar(fig, f'D06_{t}_sigma_umbrales')
    plt.show()

**(b) Curva de impacto de noticias (asimetría γ).** σ del día siguiente frente al shock
$\varepsilon_{t-1}$, partiendo de la varianza incondicional. La rama de shocks negativos
queda por encima de la de positivos si $\gamma>0$; la curva gris es el GARCH simétrico
equivalente ($\alpha+\gamma/2$ en ambas ramas).

In [ ]:
fig, axes = plt.subplots(1, len(AJUSTES_D06), figsize=(6.5 * max(len(AJUSTES_D06), 1), 4.6))
for ax, (t, a) in zip(np.atleast_1d(axes), AJUSTES_D06.items()):
    p = a['det']._params
    om, al, ga, be = (float(p[k]) for k in ('omega', 'alpha[1]', 'gamma[1]', 'beta[1]'))
    sbar = om / (1.0 - al - be - 0.5 * ga)
    eps = np.linspace(-5, 5, 401) * np.sqrt(sbar)
    gjr = np.sqrt(om + (al + ga * (eps < 0)) * eps ** 2 + be * sbar)
    sim = np.sqrt(om + (al + 0.5 * ga) * eps ** 2 + be * sbar)
    ax.plot(eps[eps < 0], gjr[eps < 0], color=viz.C_CRISIS, lw=2, label='shock negativo (α+γ)')
    ax.plot(eps[eps >= 0], gjr[eps >= 0], color=viz.C_LONG, lw=2, label='shock positivo (α)')
    ax.plot(eps, sim, color=viz.C_NA, ls='--', lw=1.2, label='simétrico (α+γ/2)')
    ax.set_xlabel('shock ε_{t−1} (retorno ×100)'); ax.set_ylabel('σ_t (×100)')
    ax.set_title(f'D06 — pista {t}: curva de impacto de noticias (γ = {ga:.3f})')
    ax.legend(loc='upper center', fontsize=8)
    i = np.argmin(np.abs(eps - 3 * np.sqrt(sbar)))
    j = np.argmin(np.abs(eps + 3 * np.sqrt(sbar)))
    print(f'Pista {t}: un shock de −3σ̄ eleva σ_t a {gjr[j]:.3f} frente a {gjr[i]:.3f} con +3σ̄ '
          f'(ratio {gjr[j] / gjr[i]:.2f}).')
fig.tight_layout()
guardar(fig, 'D06_impacto_noticias')
plt.show()

**(c) Colas: QQ-plot de los residuos estandarizados.** $z_t=(r_t-\mu)/\sigma_t$ contra una
Normal (izquierda) y contra una t de Student estandarizada con el $\nu$ estimado
(derecha). Bajo la Normal los extremos se despegan de la diagonal; bajo la t deben
alinearse: es la justificación de `dist='t'`.

In [ ]:
import scipy.stats as ss

for t, a in AJUSTES_D06.items():
    det, sigma, X = a['det'], a['sigma'], a['X']
    mu, nu = float(det._params['mu']), float(det._params['nu'])
    z = ((X['SP500_ret'] * det.scale).reindex(sigma.index) - mu) / sigma
    z = np.sort(z.dropna().values)
    q = (np.arange(1, len(z) + 1) - 0.5) / len(z)
    teo_n = ss.norm.ppf(q)
    teo_t = ss.t.ppf(q, nu) * np.sqrt((nu - 2) / nu)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    for ax, teo, nombre in ((axes[0], teo_n, 'Normal'), (axes[1], teo_t, f't de Student (ν = {nu:.2f})')):
        ax.scatter(teo, z, s=4, color=viz.C_NEG, alpha=0.5)
        lim = [min(teo.min(), z.min()), max(teo.max(), z.max())]
        ax.plot(lim, lim, color=viz.C_CRISIS, lw=1)
        ax.set_xlabel(f'cuantiles teóricos ({nombre})'); ax.set_ylabel('cuantiles empíricos de z_t')
        ax.set_title(f'D06 — pista {t}: residuos vs {nombre}')
    fig.tight_layout()
    guardar(fig, f'D06_{t}_qq_residuos')
    plt.show()
    cola = q < 0.005
    print(f'Pista {t}: curtosis (exceso) de z_t = {pd.Series(z).kurt():.2f}; en el 0,5 % inferior, '
          f'error medio |empírico − teórico| = {np.mean(np.abs(z[cola] - teo_n[cola])):.2f} (Normal) '
          f'vs {np.mean(np.abs(z[cola] - teo_t[cola])):.2f} (t).')

**(d) σ condicional frente a volatilidad realizada y distribución de σ por régimen.** La
σ del GARCH es una volatilidad *esperada*; se contrasta con una realizada *ex post*
(desviación típica móvil del retorno, misma escala). Una correlación alta confirma que la
señal que se umbraliza mide volatilidad de verdad. A la derecha, la σ separada por el
régimen que asigna el autómata, con $\tau_{in}$ y $\tau_{out}$: la banda muerta entre
ambos es la memoria que evita el parpadeo.

In [ ]:
VENTANA_RV = 21  # sesiones de la volatilidad realizada de contraste
for t, a in AJUSTES_D06.items():
    det, sigma, estados, X = a['det'], a['sigma'], a['estados'], a['X']
    rv = (X['SP500_ret'] * det.scale).rolling(VENTANA_RV).std().reindex(sigma.index)
    df = pd.DataFrame({'sigma': sigma, 'rv': rv, 'estado': estados}).dropna()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={'width_ratios': [1.2, 1]})
    cs = det.crisis_state
    for s, g in df.groupby('estado'):
        ax1.scatter(g['rv'], g['sigma'], s=3, alpha=0.35, color=viz.regime_color(int(s), 2),
                    label='crisis' if s == cs else 'calma')
    lim = [0, float(df[['sigma', 'rv']].quantile(0.999).max())]
    ax1.plot(lim, lim, color='black', lw=0.8, ls='--')
    ax1.set_xlim(lim); ax1.set_ylim(lim)
    ax1.set_xlabel(f'volatilidad realizada {VENTANA_RV}d (×100)'); ax1.set_ylabel('σ condicional (×100)')
    ax1.set_title(f'D06 — pista {t}: σ vs realizada (corr = {df.sigma.corr(df.rv):.3f})')
    ax1.legend(markerscale=4)
    viz.plot_distribution_by_regime(df['sigma'], df['estado'], crisis_state=cs,
                                    labels={0: 'calma', cs: 'crisis'}, kind='violin', ax=ax2,
                                    xlabel='σ condicional (×100)', title=f'D06 — pista {t}: σ por régimen')
    ax2.axhline(det._tau_in, color=viz.C_CRISIS, ls='--', lw=1)
    ax2.axhline(det._tau_out, color=viz.C_SHORT, ls=':', lw=1)
    fig.tight_layout()
    guardar(fig, f'D06_{t}_sigma_vs_realizada')
    plt.show()
    print(f'Pista {t}: corr(σ, vol realizada {VENTANA_RV}d) = {df.sigma.corr(df.rv):.3f}; '
          f'σ mediana calma = {df.sigma[df.estado != cs].median():.3f}, crisis = {df.sigma[df.estado == cs].median():.3f}.')

#### 4.1.3 Cobertura por crisis

Cobertura OOS con IC bootstrap, detección por evento (racha ADR-003) y *lead/lag* frente
al suelo del *drawdown*; debajo, activación en trampas no-crisis.

In [ ]:
COBERTURA = {}
for t in PISTAS:
    display(Markdown(f'#### Pista {t}'))
    COBERTURA[(t, 'D06')] = figura_cobertura(t, 'D06')
    display(COBERTURA[(t, 'D06')].round(3))
    display(tabla_trampas(t, 'D06').to_frame().round(3))

#### 4.1.4 Métricas y puesto en el ranking de detección ADR-003

In [ ]:
display(resumen_ranking(['D06']).round(4))
RESUMEN = {'D06': resumen_metricas('D06')}


def frase_ranking(d):
    for t in PISTAS:
        r = RESUMEN[d][f'pista {t}']
        print(f"Pista {t}: {d} queda {int(r['puesto_deteccion'])}º de {r['n_detectores_en_pista']} "
              f"({r['nivel_etiqueta']}), score {r['score_deteccion']:.3f}; detecta "
              f"{int(r['det_n_detectados'])}/{int(r['det_n_eventos'])} crisis; precisión {r['det_precision']:.3f} "
              f"(lift {r['det_lift_precision']:.2f}×); duración media de régimen {r['mean_regime_duration']:.1f} "
              f"sesiones; coste {r['elapsed_seconds']:.0f} s.")


frase_ranking('D06')

<a id="s4-2"></a>
### 4.2 D11 — MS-GARCH(1,1)-t (Haas-Mittnik-Paolella)

#### 4.2.1 Estados OOS y probabilidad filtrada

Aquí `p_crisis` sí es un posterior genuino: $P(S_t=\text{crisis}\mid y_{\le t})$ del
filtro de Hamilton con parámetros congelados del *train* del fold. D11 usa ventana
**rolling** y reajuste espaciado (tabla de §2) por el coste del ML en Python puro.

In [ ]:
for t in PISTAS:
    display(Markdown(f'#### Pista {t}'))
    figura_estados(t, 'D11', titulo_diag=f'D11 — pista {t}: probabilidad FILTRADA de crisis (Hamilton, causal, OOS)')
    print(f"D11/{t}: {float(es_crisis(t, 'D11').mean()):.1%} de las sesiones OOS marcadas como crisis; "
          f"{int((PANELES[(t, 'D11')]['state'].diff().fillna(0) != 0).sum())} cambios de estado.")

#### 4.2.2 Diagnóstico propio: degeneración fold a fold

El panel OOS guarda, por fecha, el **fold** que la predijo, el estado canónico y
`p_crisis`. Sin reestimar nada se obtienen tres diagnósticos por fold:

1. **nº de regímenes distintos predichos** en el bloque: si cae a 1, el modelo ha
   colapsado a un único régimen en ese bloque (el filtro no conmuta);
2. **fracción de días "crisis"** del bloque: pegada a 0 o a 1 indica un etiquetado
   económico en **esquina** (todo calma / todo crisis);
3. **dispersión intra-fold de `p_crisis`**: ≈ 0 significa posterior plano.

Las barras rojas señalan los folds problemáticos. En un bloque de pocos meses sin
turbulencia es legítimo ver un solo régimen; lo patológico es colapsar **durante** una
crisis o alternar esquinas entre folds consecutivos.

In [ ]:
UMBRAL_ESQUINA = 0.05   # fracción de crisis ≤ 5 % o ≥ 95 % = etiquetado en esquina
UMBRAL_PLANO = 0.05     # desv. típica intra-fold de p_crisis ≤ 0,05 = posterior plano


def diagnostico_folds(t, d='D11'):
    panel = PANELES[(t, d)]
    cs = estado_crisis(t, d)
    ventanas = [(pd.Timestamp(a), pd.Timestamp(b)) for a, b in rk.track_crisis_windows(t).values()]
    g = panel.groupby('fold')
    fechas = pd.Series(panel.index, index=panel.index).groupby(panel['fold'])
    diag = pd.DataFrame({
        'inicio': fechas.min(),
        'fin': fechas.max(),
        'sesiones': g.size(),
        'n_regimenes': g['state'].nunique(),
        'frac_crisis': g['state'].apply(lambda s: float((s == cs).mean())),
        'p_media': g['p_crisis'].mean(),
        'p_std': g['p_crisis'].std().fillna(0.0),
    })
    diag['solapa_crisis'] = [any(a <= f and i <= b for a, b in ventanas) for i, f in zip(diag.inicio, diag.fin)]
    diag['esquina'] = (diag.frac_crisis <= UMBRAL_ESQUINA) | (diag.frac_crisis >= 1 - UMBRAL_ESQUINA)
    diag['plano'] = diag.p_std <= UMBRAL_PLANO
    return diag


DIAG_D11 = {t: diagnostico_folds(t) for t in PISTAS}
for t, diag in DIAG_D11.items():
    etiquetas = [str(i.date())[:7] for i in diag.inicio]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7.5), sharex=True)
    viz.plot_fold_panel(etiquetas, diag.n_regimenes, ylabel='nº regímenes en el bloque', ref_line=1,
                        highlight=lambda v: v <= 1, ax=ax1,
                        title=f'D11 — pista {t}: regímenes distintos por fold (rojo = colapso a uno)')
    viz.plot_fold_panel(etiquetas, diag.frac_crisis, ylabel='fracción de días crisis', ref_line=0.5,
                        highlight=lambda v: v <= UMBRAL_ESQUINA or v >= 1 - UMBRAL_ESQUINA, ax=ax2,
                        title=f'D11 — pista {t}: fracción de crisis por fold (rojo = esquina 0 / 1)')
    for ax in (ax1, ax2):
        for k, sol in enumerate(diag.solapa_crisis):
            if sol:
                ax.axvspan(k - 0.5, k + 0.5, color=viz.C_LONG, alpha=0.10, lw=0)
    ax2.tick_params(axis='x', rotation=90, labelsize=7)
    fig.tight_layout()
    guardar(fig, f'D11_{t}_degeneracion_folds')
    plt.show()
    en_crisis = diag[diag.solapa_crisis]
    print(f'Pista {t}: {len(diag)} folds; colapsados a un régimen {float((diag.n_regimenes <= 1).mean()):.1%}; '
          f'en esquina {float(diag.esquina.mean()):.1%}; posterior plano {float(diag.plano.mean()):.1%}. '
          f'De los {len(en_crisis)} folds que solapan una crisis: '
          f'{int((en_crisis.frac_crisis <= UMBRAL_ESQUINA).sum())} en esquina "todo calma" y '
          f'{int((en_crisis.frac_crisis >= 1 - UMBRAL_ESQUINA).sum())} en "todo crisis".')
    display(en_crisis.round(3))

#### 4.2.3 Causa raíz: posterior plano y etiquetado en esquinas

Cada punto es un fold en el plano ($P(\text{crisis})$ media, fracción de días crisis),
coloreado según solape o no una ventana de crisis. Un MS-GARCH sano repartiría los folds
de crisis arriba a la derecha y los de calma abajo a la izquierda, con puntos
intermedios; un modelo degenerado los concentra en las esquinas y mezcla folds de crisis
en la esquina "todo calma" (cobertura nula aunque la volatilidad sea extrema) con folds
de calma en la esquina "todo crisis" (falsas alarmas).

In [ ]:
fig, axes = plt.subplots(1, len(PISTAS), figsize=(6.3 * len(PISTAS), 5), sharey=True)
for ax, t in zip(np.atleast_1d(axes), PISTAS):
    diag = DIAG_D11[t]
    for sol, g in diag.groupby('solapa_crisis'):
        ax.scatter(g.p_media, g.frac_crisis, s=20 + 400 * g.p_std, alpha=0.7, edgecolor='black',
                   color=viz.C_CRISIS if sol else viz.C_LONG,
                   label='fold que solapa una crisis' if sol else 'fold sin crisis')
    ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel('P(crisis) media del fold'); ax.set_ylabel('fracción de días crisis del fold')
    ax.set_title(f'D11 — pista {t} (tamaño ∝ desv. típica intra-fold de P)')
    ax.legend(loc='center right', fontsize=8)
fig.tight_layout()
guardar(fig, 'D11_causa_raiz')
plt.show()
for t in PISTAS:
    diag = DIAG_D11[t]
    print(f"Pista {t}: correlación entre fracción de crisis del fold y solape con crisis = "
          f"{diag.frac_crisis.corr(diag.solapa_crisis.astype(float)):.3f} "
          f"(un detector útil la tendría claramente positiva).")

#### 4.2.4 Opcional: ajuste de muestra completa (parámetros por régimen)

Reproduce la lectura in-sample de v1: parámetros $(\omega_k,\alpha_k,\beta_k)$, $\nu$,
matriz de transición y volatilidad incondicional por régimen. Desactivado por defecto:
el filtro de Hamilton del MS-GARCH corre en Python puro y un ajuste sobre toda la
muestra de una pista tarda minutos. Usa los parámetros del registro (los del benchmark).

In [ ]:
AJUSTE_D11_COMPLETO = False  # True: un ajuste MS-GARCH por pista sobre la muestra completa (lento)

if AJUSTE_D11_COMPLETO:
    for t in PISTAS:
        X = X_PISTA[(t, 'D11')]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            det = SPECS[(t, 'D11')].factory().fit(X)
            det.label_states_economically(X, market_returns=X['SP500_ret'])
        display(Markdown(f'#### Pista {t}'))
        display(pd.Series(det._params, name='parámetro (orden interno)').round(4).to_frame().T)
        print('vol incondicional por régimen canónico (calma, crisis), ×100:',
              np.round(det.uncond_vol_canonical(), 3))
        A = det.transition_canonical()
        print('duración esperada por régimen (sesiones):', np.round(1 / (1 - np.diag(A)), 1))
        print(f'n_parámetros = {det.n_parameters()}; logL = {det.score(X):.1f}; BIC = {det.bic(X):.1f}')
        viz.plot_transition_matrix(A, labels=['calma', 'crisis'], title=f'D11 — pista {t} (muestra completa)')
        plt.show()
else:
    print('AJUSTE_D11_COMPLETO = False: se omite el ajuste de muestra completa de D11.')

#### 4.2.5 Cobertura por crisis

In [ ]:
for t in PISTAS:
    display(Markdown(f'#### Pista {t}'))
    COBERTURA[(t, 'D11')] = figura_cobertura(t, 'D11')
    display(COBERTURA[(t, 'D11')].round(3))
    display(tabla_trampas(t, 'D11').to_frame().round(3))

#### 4.2.6 Métricas y puesto en el ranking de detección ADR-003

In [ ]:
display(resumen_ranking(['D11']).round(4))
RESUMEN['D11'] = resumen_metricas('D11')
frase_ranking('D11')

<a id="s5"></a>
## 5. Comparación intra-familia: D06 frente a D11

Misma serie (retorno del S&P 500), misma innovación t, dos maneras de convertir la
volatilidad en régimen: umbral con histéresis sobre un GARCH de un solo régimen (D06)
frente a un régimen latente con GARCH propio (D11). Se comparan (a) las métricas de
detección y persistencia, (b) la cobertura crisis a crisis y (c) cuánto coinciden sus
señales OOS día a día.

In [ ]:
cols = ['puesto_deteccion', 'nivel_etiqueta', 'score_deteccion', 'det_event_recall', 'det_precision',
        'det_lift_precision', 'det_marked_rate', 'det_mean_crisis_run', 'false_alarm_rate',
        'switching_rate', 'mean_regime_duration', 'label_stability', 'bic', 'elapsed_seconds']
COMPARA = pd.concat({d: RESUMEN[d].loc[cols] for d in DETECTORES}, axis=1)
display(COMPARA)

fig, axes = plt.subplots(len(PISTAS), 1, figsize=(14, 4.2 * len(PISTAS)))
for ax, t in zip(np.atleast_1d(axes), PISTAS):
    tab = {d: COBERTURA[(t, d)]['cobertura'] for d in DETECTORES}
    crisis = list(tab['D06'].index.union(tab['D11'].index, sort=False))
    viz.plot_grouped_bars(crisis, {d: tab[d].reindex(crisis).fillna(0).values for d in DETECTORES},
                          ylabel='cobertura OOS', title=f'Pista {t}: cobertura por crisis, D06 vs D11',
                          colors=[viz.C_LONG, viz.C_CRISIS], ax=ax, rotation=35)
    ax.set_ylim(0, 1.05)
fig.tight_layout()
guardar(fig, 'F5_comparativa_cobertura_D06_vs_D11')
plt.show()

filas = []
for t in PISTAS:
    a, b = PANELES[(t, 'D06')], PANELES[(t, 'D11')]
    idx = a.index.intersection(b.index)
    fa = a.loc[idx, 'state'] == estado_crisis(t, 'D06')
    fb = b.loc[idx, 'state'] == estado_crisis(t, 'D11')
    filas.append({
        'pista': t, 'sesiones comunes': len(idx),
        'acuerdo diario': float((fa == fb).mean()),
        'Jaccard crisis': float((fa & fb).sum() / max((fa | fb).sum(), 1)),
        'solo D06': float((fa & ~fb).mean()), 'solo D11': float((~fa & fb).mean()),
        'corr p_crisis': float(a.loc[idx, 'p_crisis'].corr(b.loc[idx, 'p_crisis'])),
    })
CONCORDANCIA = pd.DataFrame(filas).set_index('pista')
display(CONCORDANCIA.round(3))

for t in PISTAS:
    r6, r11 = RESUMEN['D06'][f'pista {t}'], RESUMEN['D11'][f'pista {t}']
    mejor = 'D06' if r6['score_deteccion'] > r11['score_deteccion'] else 'D11'
    print(f"Pista {t}: mejor score de la familia = {mejor} (D06 {r6['score_deteccion']:.3f}, "
          f"{int(r6['puesto_deteccion'])}º | D11 {r11['score_deteccion']:.3f}, {int(r11['puesto_deteccion'])}º); "
          f"D11 cuesta {r11['elapsed_seconds'] / r6['elapsed_seconds']:.1f}× el tiempo de D06; "
          f"Jaccard de sus días de crisis = {CONCORDANCIA.loc[t, 'Jaccard crisis']:.3f}.")

<a id="s6"></a>
## 6. Hallazgos de la Capa 1 (v1) y qué se mantiene / cambia en v2

**D06 en v1** (`docs/detectores/D06_garch_t_vol.md`, notebook v1 `06_garch_t_vol.ipynb`):

1. **Mecanismo justificado, no asumido**: $\gamma>0$ (apalancamiento visible en la curva de
   impacto) y residuos alineados solo bajo la t (QQ-plot); σ muy correlada con la
   volatilidad realizada.
2. **Causal y *same-day***: ocultar el futuro no cambiaba la σ del bloque; con histórico
   largo cubría la GFC y la crisis de deuda europea fuera de muestra y **captaba el
   sell-off de Q4-2018** que el HMM gaussiano (D04) apenas veía.
3. **Punto ciego en el *taper* de 2013**: shock de tipos con poca volatilidad de *equity*,
   invisible para un GARCH univariante sobre el S&P 500.
4. **Persistencia excelente** (episodios largos, poco *switching*) por histéresis + *dwell* +
   persistencia GARCH, a cambio de un `false_alarm_rate` alto frente a las ventanas
   etiquetadas de v1 (LTCM, dotcom, 2010, 2015-16, SVB… son alta volatilidad real).
5. **D06 destapó la fragilidad del etiquetado económico** (`z(std) − z(mean)` invertía
   crisis/calma cuando las medias por estado son casi iguales) y motivó el **Arreglo 4**
   del núcleo: orden **vol-primario**, hoy común a todos los detectores.

**D11 en v1** (`docs/detectores/D11_msgarch_regime.md`, notebook v1 `11_msgarch_regime.ipynb`):

1. **Implementable sin R**: HMP-2004 desde cero con filtro de Hamilton propio, causal
   (posterior del bloque idéntico al ocultar el futuro).
2. **In-sample sensato** (crisis = alta volatilidad incondicional, $\nu$ bajo, transiciones
   persistentes), pero **resultado negativo OOS**: degeneración fold a fold, con la
   paradoja de cobertura casi nula en la GFC y falsas alarmas altísimas a la vez.
3. **Causa**: ML no convexo de muchos parámetros en ventanas *rolling* cortas con un solo
   arranque; un régimen casi integrado absorbe toda la volatilidad y el otro queda muerto,
   y el etiquetado por fold se vuelve arbitrario. Conclusión v1: **argumento empírico a
   favor de la parsimonia** (D06 y D08 ganan a este modelo más expresivo).

**Qué se mantiene en v2.** Las dos clases sin cambios (movidas a
`regimenes.detectores.f5_garch`), los parámetros (umbrales y *dwell* de D06; arranques,
iteraciones, ventana *rolling* y refit espaciado de D11), el etiquetado vol-primario y la
causalidad por burn-in.

**Qué cambia en v2.** (i) Dos pistas con índice OOS común; el entrenamiento inicial de D11
pasa a ser el de la pista (tabla de §2) en vez de la ventana propia de v1; (ii) catálogo de
crisis congelado en `configs/benchmark_spec.yaml`, con Q4-2018 como **crisis** (positivo)
y el *taper* 2013 como trampa; (iii) criterio principal = **ranking de detección
ADR-003**; (iv) contexto de estado entre bloques en el walk-forward (ADR-003), que para
D06 evita que el autómata reinicie en "calma" en cada refit. Las tablas comparan v1 y v2
en métricas homologables; las ventanas no son idénticas, así que la lectura es
cualitativa.

In [ ]:
V1_DIR = rutas.DOCS / 'historia' / 'capa1' / 'resultados'
# Ventanas v1 (4 crisis + 2 trampas, ventana_eval 1993–2026) → ventana v2 más parecida.
# En v2 el sell-off de Q4-2018 es CRISIS de catálogo (positivo), no trampa.
MAPA_V1_V2 = {
    'cov_GFC_2008': 'cov_gfc_2007_09',
    'cov_EuroDebt_2011': 'cov_us_downgrade_euro2_2011',
    'cov_COVID_2020': 'cov_covid_2020',
    'cov_Inflation_2022': 'cov_inflation_bear_2022',
    'fa_Selloff_Q4_2018': 'cov_fed_tightening_2018q4',
    'fa_TaperTantrum_2013': 'fa_taper_2013',
}
COMUNES = ['ventana_eval', 'false_alarm_rate', 'switching_rate', 'mean_regime_duration',
           'label_stability', 'aic', 'bic']


def tabla_v1_v2(d, archivo_v1):
    ruta = V1_DIR / archivo_v1
    if not ruta.exists():
        print(f'No se encuentra {rel(ruta)}; se omite la comparación v1.')
        return None
    v1 = pd.read_csv(ruta).iloc[0]
    filas = []
    for c in COMUNES:
        filas.append({'métrica v1': c, 'Capa 1 (v1)': v1.get(c), 'métrica v2': c,
                      **{f'v2 pista {t}': METRICAS.loc[(t, d)].get(c) for t in PISTAS}})
    for c1, c2 in MAPA_V1_V2.items():
        filas.append({'métrica v1': c1, 'Capa 1 (v1)': v1.get(c1), 'métrica v2': c2,
                      **{f'v2 pista {t}': METRICAS.loc[(t, d)].get(c2) for t in PISTAS}})
    return pd.DataFrame(filas).set_index('métrica v1')

In [ ]:
V1 = {'D06': tabla_v1_v2('D06', 'metrics_06_garch_t_vol.csv'),
      'D11': tabla_v1_v2('D11', 'metrics_11_msgarch_regime.csv')}
for d, tabla in V1.items():
    if tabla is None:
        continue
    display(Markdown(f'#### {d}: Capa 1 (v1) frente a v2'))
    display(tabla)
    v1 = tabla['Capa 1 (v1)']
    pct = lambda v: 'fuera del OOS' if pd.isna(v) else f'{float(v):.1%}'
    print(f"{d} v1: GFC {float(v1['cov_GFC_2008']):.1%}, taper 2013 {float(v1['fa_TaperTantrum_2013']):.1%}, "
          f"Q4-2018 {float(v1['fa_Selloff_Q4_2018']):.1%}, false_alarm_rate {float(v1['false_alarm_rate']):.3f}.")
    for t in PISTAS:
        m = METRICAS.loc[(t, d)]
        print(f"{d} v2 pista {t}: GFC 2007-09 {pct(m['cov_gfc_2007_09'])}, taper 2013 {pct(m['fa_taper_2013'])}, "
              f"Q4-2018 {pct(m['cov_fed_tightening_2018q4'])}, false_alarm_rate {m['false_alarm_rate']:.3f}.")

<a id="s7"></a>
## 7. Fortalezas, limitaciones y conclusión

**Fortalezas de la familia.**
- **Causalidad nativa**: la varianza condicional solo usa el pasado; el burn-in con
  parámetros congelados la mantiene causal en el walk-forward.
- **Reacción el mismo día** a los shocks de volatilidad, reforzada por la asimetría GJR;
  colas modeladas con la t (§4.1.2).
- **D06 es barato, estable y legible**: pocos parámetros, episodios largos y una única
  señal continua (σ) con umbrales interpretables.
- **D11 aporta un posterior de régimen genuino** y una verosimilitud comparable, y
  demuestra que el MS-GARCH es abordable en Python puro.

**Limitaciones.**
- **Univariante sobre *equity***: shocks de tipos o de crédito sin volatilidad de acciones
  (tipo *taper* 2013) quedan fuera por construcción.
- **D06 no tiene estados tipados**: el régimen es un corte de σ; la `p_crisis` es una
  sigmoide, no una probabilidad calibrada, y los umbrales son percentiles del *train*.
- **Falsas alarmas frente al catálogo**: todo episodio de alta volatilidad fuera de las
  ventanas etiquetadas penaliza la precisión.
- **D11 es frágil de estimar**: ML no convexo, pocos arranques por coste, ventana *rolling*
  y refit espaciado; §4.2.2–4.2.3 muestran cuánto degenera en v2.

**Conclusión.** La celda siguiente resume, con las métricas vigentes, el veredicto de la
familia en cada pista: qué detector gana en el ranking ADR-003, cuánto degenera D11 y si
se confirma el argumento de parsimonia de la Capa 1 (el modelo simple por delante del
expresivo).

In [ ]:
lineas = []
for t in PISTAS:
    r6, r11 = RESUMEN['D06'][f'pista {t}'], RESUMEN['D11'][f'pista {t}']
    diag = DIAG_D11[t]
    parsimonia = r6['score_deteccion'] > r11['score_deteccion']
    lineas.append(
        f"- **Pista {t}**: D06 {int(r6['puesto_deteccion'])}º (score {r6['score_deteccion']:.3f}, recall "
        f"{r6['det_event_recall']:.3f}, precisión {r6['det_precision']:.3f}) | D11 {int(r11['puesto_deteccion'])}º "
        f"(score {r11['score_deteccion']:.3f}, recall {r11['det_event_recall']:.3f}, precisión "
        f"{r11['det_precision']:.3f}); folds de D11 en esquina {float(diag.esquina.mean()):.1%}. "
        f"Argumento de parsimonia (D06 > D11): {'se confirma' if parsimonia else 'NO se confirma'}.")
display(Markdown('\n'.join(lineas)))

In [ ]:
print(f'Figuras de este notebook en {rel(FIG_DIR)}:')
for ruta in sorted(FIG_DIR.glob('*.png')):
    print('  -', ruta.name)